# Discern: Strong Re-ID Backbone Training & Ablation Suite on Full Market-1501
### Proven Bag-of-Tricks (BoT) Re-ID Recipe on NVIDIA T4 GPU (MSMT17 Pretrained OSNet x1.0 & x0.5)

This ready-to-run Jupyter notebook trains a strong, compact person Re-ID backbone on the **FULL Market-1501 dataset (751 train identities)** following the proven **Bag of Tricks (Luo et al., CVPRW 2019)** recipe:

- **Backbone**: `osnet_x1_0` pretrained on MSMT17 (with `osnet_x0_5` comparison; profiles parameters and CPU/GPU latency for both).
- **Input Resolution**: $256 \times 128$.
- **Augmentations**: Random Horizontal Flip ($p=0.5$), Padding 10 + Random Crop ($256, 128$), Random Erasing ($p=0.5$).
- **Head**: BNNeck (`BatchNorm1d(512)` without bias shift).
- **Loss Function**: Cross-Entropy with Label Smoothing ($0.1$) + Batch-Hard Triplet Loss (margin $0.3$).
- **Batch Construction**: PK Sampler with $P=16$ identities, $K=4$ images ($N=64$ batch size).
- **Optimization**: Adam optimizer ($lr=3.5 \times 10^{-4}$ for backbone, $10\times = 3.5 \times 10^{-3}$ for new heads, weight decay $5 \times 10^{-4}$).
- **Schedule**: 10-epoch linear warmup followed by cosine annealing down to $10^{-6}$ over 60 epochs with Automatic Mixed Precision (AMP).
- **Model Selection**: Best checkpoint selected by **validation mAP**, not the last epoch.
- **Ablation Suite**:
  1. **(M1)**: Proven Strong Baseline (BoT recipe above).
  2. **(M2)**: + Look-Alike-Aware PK Sampler (clothing-color appearance cohorts across the 751 identities).
  3. **(M3)**: + Color-Invariance Augmentation (Random Grayscale $p=0.2$, Channel Shuffle, Color Jitter) + Look-Alike PK.
  4. **(M4)**: + Horizontal Stripe Heads (PCB spatial partitioning) + Look-Alike PK.
  5. **(M5)**: + ArcFace Loss ($s=20, m=0.25$) replacing CE (optional comparison).
  6. **(Comparison)**: `osnet_x0_5` compact baseline.
- **Evaluation**: Full standard Market-1501 protocol (CMC Rank-1, Rank-5, mAP with strict junk and same-camera filtering).

In [ ]:
# 1. Environment Verification & GPU Check
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active Device   : {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.1f} MB")

In [ ]:
# 2. Clone or Link Codebase & Install Required Packages
!pip install -q gdown scikit-learn matplotlib pandas torchvision
import os, sys

# Ensure project root is in working directory and sys.path
if os.path.exists('backend'):
    PROJECT_ROOT = os.getcwd()
elif os.path.exists('discern/backend'):
    PROJECT_ROOT = os.path.join(os.getcwd(), 'discern')
    os.chdir(PROJECT_ROOT)
else:
    !git clone https://github.com/srivardhxn/discern-openset-reid.git discern
    PROJECT_ROOT = os.path.join(os.getcwd(), 'discern')
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Active Working Directory: {os.getcwd()}")

In [ ]:
# 3. Discover Market-1501 Dataset (Kaggle Dataset or Local)
dataset_candidates = [
    '/kaggle/input/market-1501-v150915/Market-1501-v15.09.15',
    '/kaggle/input/market-1501/Market-1501-v15.09.15',
    '/kaggle/input/market1501/Market-1501-v15.09.15',
    '/kaggle/input/market1501',
    '/kaggle/input/market-1501',
    os.path.join(PROJECT_ROOT, 'data', 'Market-1501-v15.09.15'),
    os.path.join(PROJECT_ROOT, 'data', 'sample_market1501'),
]

DATA_DIR = None
for path in dataset_candidates:
    if os.path.isdir(path) and os.path.isdir(os.path.join(path, 'bounding_box_train')):
        DATA_DIR = path
        break

if DATA_DIR is None:
    print("Market-1501 not found locally. Downloading Market-1501 benchmark samples...")
    !python scripts/download_real_market1501.py
    DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'sample_market1501')

train_files = os.listdir(os.path.join(DATA_DIR, 'bounding_box_train'))
query_files = os.listdir(os.path.join(DATA_DIR, 'query')) if os.path.isdir(os.path.join(DATA_DIR, 'query')) else []
gallery_files = os.listdir(os.path.join(DATA_DIR, 'bounding_box_test')) if os.path.isdir(os.path.join(DATA_DIR, 'bounding_box_test')) else []

pids = set(f.split('_')[0] for f in train_files if not f.startswith('-1') and not f.startswith('0000'))
print(f"[DATASET FOUND] Path: {DATA_DIR}")
print(f"  - Training Images   : {len(train_files):,} ({len(pids)} unique identities)")
print(f"  - Query Images      : {len(query_files):,}")
print(f"  - Gallery Images    : {len(gallery_files):,}")

In [ ]:
# 4. Download Official MSMT17 Pretrained Weights (osnet_x1_0 and osnet_x0_5)
import urllib.request
os.makedirs('weights', exist_ok=True)

weights_map = {
    'weights/osnet_x1_0_msmt17.pth': 'https://huggingface.co/kaiyangzhou/osnet/resolve/main/osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth',
    'weights/osnet_x0_5_msmt17.pth': 'https://huggingface.co/kaiyangzhou/osnet/resolve/main/osnet_x0_5_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth',
}

for dest, url in weights_map.items():
    if not os.path.isfile(dest) or os.path.getsize(dest) < 1024*1024:
        print(f"Downloading {dest} from Hugging Face...")
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as resp, open(dest, 'wb') as out_f:
            out_f.write(resp.read())
    print(f"Verified: {dest} ({os.path.getsize(dest)/(1024*1024):.2f} MB)")

In [ ]:
# 5. Report Model Parameters and Latency (CPU & GPU) for osnet_x1_0 and osnet_x0_5
import time, torch
import pandas as pd
from backend.models.osnet import OSNetReID

def benchmark_speed(model, device, num_runs=50):
    model.eval()
    dummy = torch.randn(1, 3, 256, 128, device=device)
    with torch.no_grad():
        for _ in range(10):
            _ = model.extract_features(dummy)
            if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(num_runs):
            _ = model.extract_features(dummy)
            if device.type == 'cuda': torch.cuda.synchronize()
    latency_ms = ((time.time() - t0) / num_runs) * 1000.0
    return latency_ms, 1000.0 / latency_ms

cpu_dev = torch.device('cpu')
cuda_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

m_x1 = OSNetReID(width_mult=1.0, pretrained_path='weights/osnet_x1_0_msmt17.pth')
m_x05 = OSNetReID(width_mult=0.5, pretrained_path='weights/osnet_x0_5_msmt17.pth')

p_x1 = sum(p.numel() for p in m_x1.parameters()) / 1e6
p_x05 = sum(p.numel() for p in m_x05.parameters()) / 1e6

cpu_ms_x1, cpu_fps_x1 = benchmark_speed(m_x1.to(cpu_dev), cpu_dev)
cpu_ms_x05, cpu_fps_x05 = benchmark_speed(m_x05.to(cpu_dev), cpu_dev)

gpu_ms_x1, gpu_fps_x1 = benchmark_speed(m_x1.to(cuda_dev), cuda_dev) if torch.cuda.is_available() else (None, None)
gpu_ms_x05, gpu_fps_x05 = benchmark_speed(m_x05.to(cuda_dev), cuda_dev) if torch.cuda.is_available() else (None, None)

prof_df = pd.DataFrame([
    {
        'Backbone': 'OSNet x1.0 (Standard)',
        'Parameters (M)': f"{p_x1:.3f}M",
        'CPU Latency': f"{cpu_ms_x1:.1f} ms ({cpu_fps_x1:.1f} FPS)",
        'GPU Latency (T4)': f"{gpu_ms_x1:.2f} ms ({gpu_fps_x1:.0f} FPS)" if gpu_ms_x1 else 'N/A',
    },
    {
        'Backbone': 'OSNet x0.5 (Compact)',
        'Parameters (M)': f"{p_x05:.3f}M",
        'CPU Latency': f"{cpu_ms_x05:.1f} ms ({cpu_fps_x05:.1f} FPS)",
        'GPU Latency (T4)': f"{gpu_ms_x05:.2f} ms ({gpu_fps_x05:.0f} FPS)" if gpu_ms_x05 else 'N/A',
    }
])
display(prof_df)

In [ ]:
# 6. Train BoT Re-ID Ablation Models on Full Market-1501 (60 Epochs, AMP, Best Checkpoint Selection)
!python scripts/kaggle_train.py \
    --data-dir "{DATA_DIR}" \
    --epochs 60 \
    --batch-p 16 \
    --batch-k 4 \
    --lr 0.00035 \
    --val-interval 5 \
    --device cuda \
    --run-ablation \
    --include-arcface \
    --include-x05

In [ ]:
# 7. Run Open-Set Scientific Evaluation Suite with 95% Bootstrap Confidence Intervals
!python scripts/evaluate.py --data-dir "{DATA_DIR}"

In [ ]:
# 8. Display Verified Benchmark Results, Ablation Table, and ROC Plots
import json
import pandas as pd
from IPython.display import Image, display

manifest_path = 'results/kaggle_training_manifest.json'
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    print("\n=================== FULL MARKET-1501 STANDARD BENCHMARK RESULTS ===================")
    table_rows = []
    for m in manifest.get('models', []):
        std = m.get('final_market1501', {})
        prof = m.get('profiling', {})
        table_rows.append({
            'Model Configuration': m.get('model_name'),
            'Backbone': m.get('backbone'),
            'Rank-1 (%)': f"{std.get('rank1', 0.0)*100:.2f}%",
            'Rank-5 (%)': f"{std.get('rank5', 0.0)*100:.2f}%",
            'mAP (%)': f"{std.get('map', 0.0)*100:.2f}%",
            'Best Epoch': m.get('best_epoch'),
            'Train Time': f"{m.get('training_time_seconds', 0.0):.1f}s",
            'Params': f"{prof.get('parameters_million', 0.0)}M",
        })
    res_df = pd.DataFrame(table_rows)
    display(res_df)

eval_path = 'results/evaluation_results.json'
if os.path.exists(eval_path):
    with open(eval_path, 'r') as f:
        ev = json.load(f)
    print("\n=================== OPEN-SET PROTOCOL ABLATION RESULTS ===================")
    open_rows = []
    for r in ev.get('ablation_study', []):
        open_rows.append({
            'Component': r['component'],
            'Full TAR @ 1% FAR': f"{r['full_tar_1pct']*100:.1f}% [{r['full_tar_1pct_ci'][0]*100:.1f}%, {r['full_tar_1pct_ci'][1]*100:.1f}%]",
            'Full DIR @ 1% FAR': f"{r['full_dir_1pct']*100:.1f}% [{r['full_dir_1pct_ci'][0]*100:.1f}%, {r['full_dir_1pct_ci'][1]*100:.1f}%]",
            'LowVar TAR @ 1% FAR': f"{r['lowvar_tar_1pct']*100:.1f}% [{r['lowvar_tar_1pct_ci'][0]*100:.1f}%, {r['lowvar_tar_1pct_ci'][1]*100:.1f}%]",
            'LowVar DIR @ 1% FAR': f"{r['lowvar_dir_1pct']*100:.1f}% [{r['lowvar_dir_1pct_ci'][0]*100:.1f}%, {r['lowvar_dir_1pct_ci'][1]*100:.1f}%]",
        })
    display(pd.DataFrame(open_rows))

if os.path.exists('results/roc_chart.png'):
    display(Image('results/roc_chart.png'))